# Multi-Label Image Classification — Computer Vision Project

| | |
|---|---|
| **Course** | Computer Vision — BS Computer Science, Semester VIII |
| **Framework** | TensorFlow / Keras |
| **Task** | Multi-Label Image Classification (Pascal VOC 2012) |
| **Models** | VGG16 · ResNet50 · MobileNetV2 |
| **Group Members** | Rajeev Kumar — 023-22-0145 |
| | Darshan Jethani — 023-22-0088 |
| | Zahid Hussain — 023-22-0053 |

---

## Notebook Structure

| Section | Content |
|---|---|
| 1 | Dataset details |
| 2 | Data loading |
| 3 | Preprocessing & augmentation |
| 4 | Model architecture |
| 5 | Part A — Backbone comparison (3 models, full regularization) |
| 6 | Part B — Ablation study (ResNet50 × 5 regularization configs) |
| 7 | Results & comparative analysis |
| 8 | Conclusion |

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP — Run this cell first!
# ════════════════════════════════════════════════════════════════════════════════
# 
# 📌 IMPORTANT: This notebook is designed to run on Kaggle.
#    If running LOCALLY, uncomment the pip install line below:

import sys
if '/kaggle' not in sys.prefix:
    # Uncomment the next line to install required packages locally
    # %pip install tensorflow scikit-learn pandas matplotlib -q
    print("⚠️  Running locally. If you see import errors, uncomment the pip install line above.")
else:
    print("✅ Running on Kaggle - all packages pre-installed.")

## Section 1 — Dataset Details

**Dataset:** Pascal VOC 2012 — `huanghanchina/pascal-voc-2012` on Kaggle

| Property | Value |
|---|---|
| Total images | ~11,540 |
| Classes | 20 object categories |
| Task type | Multi-label — one image can contain multiple classes |
| Train split | official `train.txt` (~8,498 images) |
| Validation split | first 50% of `val.txt` |
| Test split | second 50% of `val.txt` |

**20 Classes:** aeroplane, bicycle, bird, boat, bottle, bus, car, cat, chair, cow, diningtable, dog, horse, motorbike, person, pottedplant, sheep, sofa, train, tvmonitor

**Why multi-label?** Each image contains 2–4 objects on average. Every class is treated as an independent yes/no question → sigmoid activation + binary crossentropy loss.


In [2]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    level = root.replace('/kaggle/input', '').count(os.sep)
    if level < 3:
        print('  ' * level + os.path.basename(root) + '/')

In [ ]:
# ─────────────────────────────────────────────────────────────
# SECTION 1 — IMPORTS
# ─────────────────────────────────────────────────────────────
# 📌 NOTE: Ensure the following packages are installed:
#    TensorFlow, scikit-learn, pandas, matplotlib, numpy
#    Run the ENVIRONMENT SETUP cell above if imports fail.

import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET

# TensorFlow & Keras imports - Required for deep learning models
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, regularizers
    from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2
except ImportError as e:
    print(f"❌ ERROR: {e}")
    print("Please install TensorFlow: %pip install tensorflow")
    raise

# ── Per-backbone preprocessing functions ─────────────────────
# Each backbone was trained on ImageNet using its own pixel normalization.
# Applying the wrong preprocessing (or none) degrades frozen feature quality.
from tensorflow.keras.applications.vgg16      import preprocess_input as vgg_preprocess
from tensorflow.keras.applications.resnet50   import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobile_preprocess

from tensorflow.keras.callbacks import (EarlyStopping, ModelCheckpoint,
                                         ReduceLROnPlateau)

# Scikit-learn metrics for multi-label evaluation
try:
    from sklearn.metrics import (f1_score, precision_score,
                                  recall_score, accuracy_score)
except ImportError:
    print("❌ ERROR: scikit-learn not installed. Run: %pip install scikit-learn")
    raise

warnings.filterwarnings('ignore')
print("TensorFlow :", tf.__version__)
print("GPU        :", tf.config.list_physical_devices('GPU'))

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# ─────────────────────────────────────────────────────────────
# CONFIGURATION — all hyperparameters in one place
# ─────────────────────────────────────────────────────────────

IMG_SIZE     = 224      # Standard input size for VGG16 / ResNet50 / MobileNetV2
BATCH_SIZE   = 32       # Mini-batch size
EPOCHS_MAIN  = 50       # Max epochs — Part A (Early Stopping will stop sooner)
EPOCHS_ABL   = 30       # Max epochs — ablation runs without Early Stopping
LR           = 1e-4     # Adam initial learning rate
DROPOUT_1    = 0.5      # Dropout rate after Dense(512)
DROPOUT_2    = 0.3      # Dropout rate after Dense(256)
L2_LAMBDA    = 1e-4     # L2 weight-decay coefficient
PATIENCE     = 7        # Early Stopping patience (epochs with no val_loss improvement)
NUM_CLASSES  = 20       # Pascal VOC 2012 object classes
THRESHOLD    = 0.5      # Sigmoid prediction threshold
SEED         = 42       # Fixed seed for reproducibility

# Maps each backbone name to its official Keras preprocessing function.
# These functions convert raw [0,255] pixels to the exact range each
# backbone's frozen BatchNorm layers expect.
PREPROCESS_MAP = {
    'VGG16':       vgg_preprocess,     # subtracts ImageNet channel means
    'ResNet50':    resnet_preprocess,  # subtracts ImageNet channel means
    'MobileNetV2': mobile_preprocess,  # scales to [-1, 1]
}

VOC_CLASSES = [
    'aeroplane','bicycle','bird','boat','bottle',
    'bus','car','cat','chair','cow',
    'diningtable','dog','horse','motorbike','person',
    'pottedplant','sheep','sofa','train','tvmonitor'
]

# ── Dataset paths ──────────────────────────────────────────────
# 📌 IMPORTANT: This notebook expects the Pascal VOC 2012 dataset
#
# ✅ ON KAGGLE: Add dataset 'huanghanchina/pascal-voc-2012' via Data panel
# ❌ LOCALLY: Download the dataset and update BASE_DIR below
#
# Local setup example:
#   BASE_DIR = './data/VOC2012'  # Replace with your local path

BASE_DIR = '/kaggle/input/pascal-voc-2012/VOC2012'
IMG_DIR  = os.path.join(BASE_DIR, 'JPEGImages')
ANN_DIR  = os.path.join(BASE_DIR, 'Annotations')
SETS_DIR = os.path.join(BASE_DIR, 'ImageSets', 'Main')

# Verify paths exist
print("\n📂 Checking dataset paths:")
for p in [IMG_DIR, ANN_DIR, SETS_DIR]:
    exists = os.path.exists(p)
    status = '✅' if exists else '❌'
    print(f"{status}  {p}")
    
if not all(os.path.exists(p) for p in [IMG_DIR, ANN_DIR, SETS_DIR]):
    print("\n⚠️  WARNING: Some paths not found!")
    print("   If running locally, update BASE_DIR with your dataset location.")

NameError: name 'vgg_preprocess' is not defined

## Section 2 — Data Loading

In [ ]:
# ─────────────────────────────────────────────────────────────
# PARSE VOC XML ANNOTATIONS → multi-hot label vectors
# ─────────────────────────────────────────────────────────────

def parse_annotation(xml_path):
    """
    Read a Pascal VOC XML file and return a float32 array of shape (20,).
    Position i is 1.0 if class VOC_CLASSES[i] appears in the image, else 0.0.
    """
    tree  = ET.parse(xml_path)
    label = np.zeros(NUM_CLASSES, dtype=np.float32)
    for obj in tree.getroot().findall('object'):
        name = obj.find('name').text.strip()
        if name in VOC_CLASSES:
            label[VOC_CLASSES.index(name)] = 1.0
    return label


def load_split(split_name):
    """Load all image paths + labels for one official Pascal VOC split."""
    ids = open(os.path.join(SETS_DIR, f'{split_name}.txt')).read().split()
    paths, labels = [], []
    for img_id in ids:
        ip = os.path.join(IMG_DIR, f'{img_id}.jpg')
        ap = os.path.join(ANN_DIR, f'{img_id}.xml')
        if os.path.exists(ip) and os.path.exists(ap):
            paths.append(ip)
            labels.append(parse_annotation(ap))
    print(f'  [{split_name}] {len(paths):,} images loaded')
    return paths, np.array(labels, dtype=np.float32)


print('Loading Pascal VOC 2012 ...')
train_paths, train_labels = load_split('train')
all_val_paths, all_val_labels = load_split('val')

# Split val → 50% validation + 50% test (SEED fixes the split for reproducibility)
np.random.seed(SEED)
idx  = np.random.permutation(len(all_val_paths))
half = len(idx) // 2

val_paths   = [all_val_paths[i] for i in idx[:half]]
val_labels  = all_val_labels[idx[:half]]
test_paths  = [all_val_paths[i] for i in idx[half:]]
test_labels = all_val_labels[idx[half:]]

print(f'\nFinal split:')
print(f'  Train      : {len(train_paths):,}')
print(f'  Validation : {len(val_paths):,}')
print(f'  Test       : {len(test_paths):,}')
print(f'  Total      : {len(train_paths)+len(val_paths)+len(test_paths):,}')


## Section 3 — Data Preprocessing & Augmentation

**Key design decision:** Images are loaded as raw `float32` in the range **[0, 255]**.  
Each model applies its own `preprocess_input` function internally (inside `build_model`).  
This ensures every backbone receives pixels in the exact range its frozen  
BatchNorm layers were calibrated on during ImageNet training.

| Backbone | What `preprocess_input` does |
|---|---|
| VGG16 | Subtracts ImageNet channel means: R=103.9, G=116.8, B=123.7 |
| ResNet50 | Subtracts ImageNet channel means (same formula, different values) |
| MobileNetV2 | Scales to [−1, 1]: `pixel / 127.5 − 1` |

Augmentation is applied **only on training images** and works on the [0, 255] scale.


In [ ]:
# ─────────────────────────────────────────────────────────────
# DATA PIPELINE — load raw [0,255] images, augment training only
# ─────────────────────────────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

@tf.function
def load_image(path):
    """
    Read JPEG → decode RGB → resize 224×224 → cast to float32.
    Pixel values stay in [0, 255]. Each backbone's preprocess_input
    is applied INSIDE the model, not here, so one dataset serves all models.
    """
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return tf.cast(img, tf.float32)   # range [0, 255]


@tf.function
def augment(image, label):
    """
    Random augmentations (training set only). All operations work on
    the [0, 255] pixel scale — brightness delta and clip are scaled accordingly.

    Techniques applied:
      1. Random horizontal flip
      2. Random brightness  (±50 out of 255, roughly ±20%)
      3. Random contrast    (80%–120%)
      4. Random saturation  (80%–120%)
      5. Random crop        (pad 20px then crop back to 224×224)
    """
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=50.0)   # ±50/255 ≈ ±20%
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    image = tf.image.random_saturation(image, lower=0.8, upper=1.2)
    padded = tf.image.resize_with_crop_or_pad(image, IMG_SIZE+20, IMG_SIZE+20)
    image  = tf.image.random_crop(padded, [IMG_SIZE, IMG_SIZE, 3], seed=SEED)
    image  = tf.clip_by_value(image, 0.0, 255.0)                # stay in [0,255]
    return image, label


def make_dataset(paths, labels, aug=False, shuffle=False):
    """Build an efficient tf.data pipeline (augmentation = training only)."""
    ds = tf.data.Dataset.zip((
        tf.data.Dataset.from_tensor_slices(paths),
        tf.data.Dataset.from_tensor_slices(labels)
    ))
    if shuffle:
        ds = ds.shuffle(2048, seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, l: (load_image(p), l), num_parallel_calls=AUTOTUNE)
    if aug:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_ds = make_dataset(train_paths, train_labels, aug=True,  shuffle=True)
val_ds   = make_dataset(val_paths,   val_labels,   aug=False, shuffle=False)
test_ds  = make_dataset(test_paths,  test_labels,  aug=False, shuffle=False)

print('✅ tf.data pipelines ready')
print(f'  Train batches : {len(train_ds)}')
print(f'  Val batches   : {len(val_ds)}')
print(f'  Test batches  : {len(test_ds)}')


In [ ]:
# ─────────────────────────────────────────────────────────────
# VISUALIZE sample images + class distribution
# Note: divide by 255 for display since pixels are stored as [0,255]
# ─────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for imgs, lbls in train_ds.take(1):
    for i, ax in enumerate(axes.flat):
        ax.imshow(imgs[i].numpy() / 255.0)   # divide by 255 for display only
        names = [VOC_CLASSES[j] for j in range(NUM_CLASSES) if lbls[i][j] == 1]
        ax.set_title(', '.join(names), fontsize=8)
        ax.axis('off')
plt.suptitle('Training samples (augmented, displayed as [0,1])',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Class distribution bar chart
counts   = train_labels.sum(axis=0)
idx_sort = np.argsort(counts)[::-1]
plt.figure(figsize=(14, 4))
plt.bar([VOC_CLASSES[i] for i in idx_sort],
        [counts[i] for i in idx_sort], color='steelblue')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.title('Class distribution — training set')
plt.tight_layout(); plt.show()
print(f'Average labels per image: {train_labels.sum(axis=1).mean():.2f}')


## Section 4 — Model Architecture

### Transfer learning setup

Backbone weights loaded from **ImageNet** (`include_top=False`, `trainable=False`).  
Only the custom classification head is trained.

### Classification head (identical for all 3 backbones)

```
Input [0,255]
  → preprocess_input(backbone)   ← model-specific pixel normalization
  → Frozen backbone              ← ImageNet feature extractor
  → GlobalAveragePooling2D       ← (7×7×C) feature map → (C,) vector
  → Dense(512, relu, [L2])       → [Dropout(0.5)]
  → Dense(256, relu, [L2])       → [Dropout(0.3)]
  → Dense(20, sigmoid)           ← independent probability per class
```

**Why sigmoid (not softmax)?**  
Softmax forces all 20 outputs to sum to 1 — only one class can win.  
Sigmoid gives each class an independent probability — multiple can be active at once.

**Why binary_crossentropy?**  
Treats each of the 20 labels as a separate binary problem and averages their losses.


In [ ]:
# ─────────────────────────────────────────────────────────────
# MODEL BUILDER — preprocess_input applied INSIDE the model
# ─────────────────────────────────────────────────────────────

def build_model(backbone_name: str,
                use_dropout:   bool,
                use_l2:        bool) -> keras.Model:
    """
    Build a multi-label classifier with a frozen pretrained backbone.

    The first operation inside the model is the backbone-specific
    preprocess_input, which converts raw [0,255] pixels to the exact
    range the frozen BatchNorm layers were calibrated on during
    ImageNet pre-training.

    Parameters
    ----------
    backbone_name : 'VGG16' | 'ResNet50' | 'MobileNetV2'
    use_dropout   : add Dropout(0.5) and Dropout(0.3) to the head
    use_l2        : add L2(1e-4) weight-decay to Dense layers
    """
    reg    = regularizers.l2(L2_LAMBDA) if use_l2 else None
    kwargs = dict(weights='imagenet', include_top=False,
                  input_shape=(IMG_SIZE, IMG_SIZE, 3))

    if   backbone_name == 'VGG16':       base = VGG16(**kwargs)
    elif backbone_name == 'ResNet50':    base = ResNet50(**kwargs)
    elif backbone_name == 'MobileNetV2': base = MobileNetV2(**kwargs)
    else: raise ValueError(f'Unknown backbone: {backbone_name}')

    base.trainable = False  # freeze all backbone weights

    inp = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='image_input')

    # ── Apply backbone-specific preprocessing ────────────────
    # This is the critical fix: each backbone needs its own pixel
    # normalization. Without it, frozen BatchNorm statistics are mismatched.
    preprocess_fn = PREPROCESS_MAP[backbone_name]
    x = layers.Lambda(
        lambda img: preprocess_fn(img),
        name='preprocess_input'
    )(inp)

    # ── Frozen backbone feature extraction ───────────────────
    x = base(x, training=False)   # training=False keeps BN frozen

    # ── Classification head ──────────────────────────────────
    x = layers.GlobalAveragePooling2D(name='gap')(x)

    x = layers.Dense(512, activation='relu',
                      kernel_regularizer=reg, name='dense_512')(x)
    if use_dropout:
        x = layers.Dropout(DROPOUT_1, name='drop_1')(x)

    x = layers.Dense(256, activation='relu',
                      kernel_regularizer=reg, name='dense_256')(x)
    if use_dropout:
        x = layers.Dropout(DROPOUT_2, name='drop_2')(x)

    # Sigmoid: each of the 20 outputs is independent (multi-label)
    out = layers.Dense(NUM_CLASSES, activation='sigmoid',
                        name='predictions')(x)

    model = keras.Model(inp, out, name=backbone_name)
    model.compile(
        optimizer=keras.optimizers.Adam(LR),
        loss='binary_crossentropy',   # correct loss for multi-label
        metrics=['accuracy',
                 keras.metrics.Precision(name='precision'),
                 keras.metrics.Recall(name='recall')]
    )
    return model


# Verify: print trainable parameter counts
print('Trainable parameters (head only — backbone frozen):')
for name in ['VGG16', 'ResNet50', 'MobileNetV2']:
    m  = build_model(name, use_dropout=True, use_l2=True)
    tp = sum(tf.size(v).numpy() for v in m.trainable_variables)
    tt = sum(tf.size(v).numpy() for v in m.variables)
    print(f'  {name:<15}: {tp:>10,} trainable / {tt:>12,} total')
    del m; keras.backend.clear_session()


## Section 5 — Training

In [ ]:
# ─────────────────────────────────────────────────────────────
# TRAINING UTILITIES
# ─────────────────────────────────────────────────────────────

def get_callbacks(run_name: str, use_early_stopping: bool) -> list:
    """
    Build Keras callbacks for one training run.

    EarlyStopping  — stops when val_loss does not improve for PATIENCE epochs,
                     then restores model weights from the best epoch.
    ModelCheckpoint— saves the best weights to disk (backup).
    ReduceLROnPlateau— halves learning rate after 3 stagnant epochs (min 1e-7).
    """
    cbs = [
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=3, min_lr=1e-7, verbose=1),
        ModelCheckpoint(f'{run_name}_best.keras',
                        monitor='val_loss', save_best_only=True, verbose=0),
    ]
    if use_early_stopping:
        cbs.append(
            EarlyStopping(monitor='val_loss', patience=PATIENCE,
                          restore_best_weights=True, verbose=1)
        )
    return cbs


def train_model(model, run_name: str,
                max_epochs: int,
                use_early_stopping: bool = True):
    """Run model.fit() and return (history, elapsed_seconds)."""
    has_do = any('drop' in l.name for l in model.layers)
    has_l2 = any(hasattr(l, 'kernel_regularizer') and l.kernel_regularizer
                 for l in model.layers)
    print(f'\n{"="*58}')
    print(f'  Run            : {run_name}')
    print(f'  Dropout        : {"YES" if has_do else "NO"}')
    print(f'  L2             : {"YES" if has_l2 else "NO"}')
    print(f'  Early Stopping : {"YES" if use_early_stopping else "NO"}')
    print(f'  Max epochs     : {max_epochs}')
    print(f'{"="*58}')

    t0 = time.time()
    hist = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=max_epochs,
        callbacks=get_callbacks(run_name, use_early_stopping),
        verbose=1
    )
    elapsed = time.time() - t0
    print(f'  Done: {elapsed/60:.1f} min | Epochs run: {len(hist.epoch)}')
    return hist, elapsed


def evaluate_model(model, dataset, label: str) -> dict:
    """
    Evaluate model on dataset.
    Returns Accuracy, Precision, Recall, F1-Micro, F1-Macro
    using THRESHOLD=0.5 on sigmoid outputs.
    """
    yt, yp = [], []
    for imgs, lbls in dataset:
        preds = model.predict(imgs, verbose=0)
        yt.append(lbls.numpy())
        yp.append((preds >= THRESHOLD).astype(np.int32))
    yt = np.vstack(yt); yp = np.vstack(yp)
    return {
        'Run'      : label,
        'Accuracy' : float(accuracy_score(yt.ravel(), yp.ravel())),
        'Precision': float(precision_score(yt, yp, average='micro', zero_division=0)),
        'Recall'   : float(recall_score(yt, yp, average='micro', zero_division=0)),
        'F1_Micro' : float(f1_score(yt, yp, average='micro', zero_division=0)),
        'F1_Macro' : float(f1_score(yt, yp, average='macro', zero_division=0)),
    }

print('✅ Training utilities ready')


### Part A — Backbone Comparison

All three backbones trained with **full regularization**:  
`Dropout=True` · `L2=True` · `Early Stopping=True`

This answers: *which backbone architecture performs best on Pascal VOC 2012?*


In [ ]:
# ─────────────────────────────────────────────────────────────
# PART A — BACKBONE COMPARISON
# VGG16 · ResNet50 · MobileNetV2  (all with full regularization)
# ─────────────────────────────────────────────────────────────

backbone_histories = {}
backbone_results   = []

for backbone in ['VGG16', 'ResNet50', 'MobileNetV2']:

    model = build_model(backbone, use_dropout=True, use_l2=True)

    hist, elapsed = train_model(model,
                                run_name=f'partA_{backbone}',
                                max_epochs=EPOCHS_MAIN,
                                use_early_stopping=True)

    backbone_histories[backbone] = hist

    metrics = evaluate_model(model, test_ds, label=backbone)
    metrics['Epochs_Run'] = len(hist.epoch)
    metrics['Time_min']   = round(elapsed / 60, 1)
    backbone_results.append(metrics)

    model.save(f'{backbone}_full_reg.keras')
    del model; keras.backend.clear_session()

print('\n✅ Part A complete — all 3 backbones trained')


### Part B — Ablation Study (ResNet50)

We train **ResNet50** in 5 configurations — one technique added at a time.  
This isolates the contribution of each regularization technique.

| Exp | Dropout | Early Stopping | Weight Decay (L2) | Purpose |
|---|:---:|:---:|:---:|---|
| 1 — Baseline | ❌ | ❌ | ❌ | No regularization — reference point |
| 2 — Dropout only | ✅ | ❌ | ❌ | Isolate Dropout's effect |
| 3 — Early Stopping only | ❌ | ✅ | ❌ | Isolate Early Stopping's effect |
| 4 — L2 only | ❌ | ❌ | ✅ | Isolate Weight Decay's effect |
| 5 — Full model | ✅ | ✅ | ✅ | All three combined |

**Why ResNet50 for ablation?** It is the middle-ground model — more complex than MobileNetV2 (more likely to overfit) but lighter than VGG16 (faster per epoch). This makes regularization effects clearly visible.

**GPU time note:** Experiments without Early Stopping use `EPOCHS_ABL=30` max to stay within Kaggle's free GPU limit.


In [ ]:
# ─────────────────────────────────────────────────────────────
# PART B — ABLATION STUDY (ResNet50 × 5 configurations)
# ─────────────────────────────────────────────────────────────

# (label, use_dropout, use_l2, use_early_stopping)
ablation_configs = [
    ('Exp1-Baseline',      False, False, False),
    ('Exp2-DropoutOnly',   True,  False, False),
    ('Exp3-EarlyStpOnly',  False, False, True ),
    ('Exp4-L2Only',        False, True,  False),
    ('Exp5-FullModel',     True,  True,  True ),
]

ablation_histories = {}
ablation_results   = []

for exp_label, use_do, use_l2, use_es in ablation_configs:

    model = build_model('ResNet50', use_dropout=use_do, use_l2=use_l2)

    # Experiments with Early Stopping can safely use 50 epochs (ES guards it).
    # Experiments without Early Stopping are capped at 30 to save GPU time.
    max_ep = EPOCHS_MAIN if use_es else EPOCHS_ABL

    hist, elapsed = train_model(model,
                                run_name=f'abl_{exp_label}',
                                max_epochs=max_ep,
                                use_early_stopping=use_es)

    ablation_histories[exp_label] = hist

    metrics = evaluate_model(model, test_ds, label=exp_label)
    metrics['Dropout']       = 'Yes' if use_do else 'No'
    metrics['EarlyStopping'] = 'Yes' if use_es else 'No'
    metrics['L2_Decay']      = 'Yes' if use_l2 else 'No'
    metrics['Epochs_Run']    = len(hist.epoch)
    metrics['Time_min']      = round(elapsed / 60, 1)
    ablation_results.append(metrics)

    del model; keras.backend.clear_session()

print('\n✅ Part B complete — all 5 ablation experiments done')


## Section 6 — Results & Comparative Analysis

In [ ]:
# ─────────────────────────────────────────────────────────────
# RESULTS TABLES
# ─────────────────────────────────────────────────────────────

df_backbone = pd.DataFrame(backbone_results)
df_ablation = pd.DataFrame(ablation_results)
metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1_Micro', 'F1_Macro']

print('='*68)
print('  PART A — Backbone Comparison (full regularization)')
print('='*68)
print(df_backbone[['Run'] + metric_cols + ['Epochs_Run','Time_min']]
      .to_string(index=False, float_format='{:.4f}'.format))

print('\n' + '='*68)
print('  PART B — Ablation Study (ResNet50, one technique at a time)')
print('='*68)
print(df_ablation[['Run','Dropout','EarlyStopping','L2_Decay']
                  + metric_cols + ['Epochs_Run']]
      .to_string(index=False, float_format='{:.4f}'.format))


In [ ]:
# ─────────────────────────────────────────────────────────────
# PLOT 1 — Training & Validation Curves (Part A — 3 backbones)
# ─────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
PAL = {'train': '#2196F3', 'val': '#FF5722'}

for col, name in enumerate(['VGG16', 'ResNet50', 'MobileNetV2']):
    h = backbone_histories[name].history

    ax = axes[0, col]
    ax.plot(h['loss'],     color=PAL['train'], lw=2, label='Train')
    ax.plot(h['val_loss'], color=PAL['val'],   lw=2, ls='--', label='Val')
    ax.set_title(f'{name} — Loss', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Binary crossentropy')
    ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1, col]
    ax.plot(h['accuracy'],     color=PAL['train'], lw=2, label='Train')
    ax.plot(h['val_accuracy'], color=PAL['val'],   lw=2, ls='--', label='Val')
    ax.set_title(f'{name} — Accuracy', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Part A — Training Curves (Full Regularization)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('plot1_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────
# PLOT 2 — Backbone Metric Comparison Bar Chart (Part A)
# ─────────────────────────────────────────────────────────────

metrics_plot = ['Accuracy', 'Precision', 'Recall', 'F1_Micro', 'F1_Macro']
x      = np.arange(len(metrics_plot))
width  = 0.22
colors = ['#2196F3', '#4CAF50', '#FF9800']

fig, ax = plt.subplots(figsize=(13, 5))
for i, (row, color) in enumerate(zip(df_backbone.itertuples(), colors)):
    vals = [getattr(row, m) for m in metrics_plot]
    bars = ax.bar(x + i * width, vals, width, label=row.Run,
                  color=color, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(x + width)
ax.set_xticklabels(metrics_plot, fontsize=11)
ax.set_ylim(0, 1.12); ax.set_ylabel('Score')
ax.set_title('Part A — VGG16 vs ResNet50 vs MobileNetV2 (Full Regularization)',
             fontsize=13, fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('plot2_backbone_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────
# PLOT 3 — Ablation: Val-Loss curves + F1 bar chart (ResNet50)
# ─────────────────────────────────────────────────────────────

abl_colors  = ['#9E9E9E', '#2196F3', '#FF9800', '#9C27B0', '#4CAF50']
exp_display = ['Baseline', 'Dropout only', 'EarlyStp only', 'L2 only', 'Full model']
exp_keys    = [cfg[0] for cfg in ablation_configs]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: validation loss over epochs
ax = axes[0]
for key, color, disp in zip(exp_keys, abl_colors, exp_display):
    h = ablation_histories[key].history
    ax.plot(h['val_loss'], label=disp, color=color, lw=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation loss')
ax.set_title('Ablation — Validation Loss per Experiment', fontweight='bold')
ax.legend(fontsize=9, loc='upper right'); ax.grid(alpha=0.3)

# Right: F1-Micro bar chart
ax = axes[1]
f1_vals = [r['F1_Micro'] for r in ablation_results]
bars    = ax.barh(exp_display, f1_vals, color=abl_colors, edgecolor='white')
for bar, v in zip(bars, f1_vals):
    ax.text(v + 0.003, bar.get_y() + bar.get_height()/2,
            f'{v:.4f}', va='center', fontsize=10, fontweight='bold')
ax.axvline(x=f1_vals[-1], color='green', ls='--', lw=1.5,
           label=f'Full model: {f1_vals[-1]:.3f}')
ax.set_xlabel('F1-Score (micro)')
ax.set_title('Ablation — F1-Micro Comparison', fontweight='bold')
ax.set_xlim(0, 1.05); ax.legend(fontsize=9); ax.grid(axis='x', alpha=0.3)

plt.suptitle('Part B — Ablation Study: ResNet50 (one regularization at a time)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot3_ablation_study.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────
# PLOT 4 — Per-class F1 for the best backbone model
# Fix: rebuild model architecture then load weights only
# (avoids Lambda layer deserialization error)
# ─────────────────────────────────────────────────────────────

best_name  = df_backbone.loc[df_backbone['F1_Micro'].idxmax(), 'Run']
print(f'Best backbone: {best_name}')

# Rebuild the model architecture fresh, then load saved weights into it
best_model = build_model(best_name, use_dropout=True, use_l2=True)
best_model.load_weights(f'partA_{best_name}_best.keras')

yt_all, yp_all = [], []
for imgs, lbls in test_ds:
    preds = best_model.predict(imgs, verbose=0)
    yt_all.append(lbls.numpy())
    yp_all.append((preds >= THRESHOLD).astype(np.int32))
yt_all = np.vstack(yt_all); yp_all = np.vstack(yp_all)

per_class = f1_score(yt_all, yp_all, average=None, zero_division=0)
df_cls    = (pd.DataFrame({'Class': VOC_CLASSES, 'F1': per_class})
               .sort_values('F1', ascending=True))

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(df_cls['Class'], df_cls['F1'],
        color=['#f44336' if v < 0.5 else '#4CAF50' for v in df_cls['F1']])
ax.axvline(x=0.5, color='black', ls='--', lw=1.2, label='0.5 threshold')
ax.set_xlabel('F1-Score'); ax.set_xlim(0, 1)
ax.set_title(f'Per-Class F1-Score — {best_name} (best model)',
             fontsize=13, fontweight='bold')
ax.legend(); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('plot4_per_class_f1.png', dpi=150, bbox_inches='tight')
plt.show()

del best_model; keras.backend.clear_session()

In [ ]:
# ─────────────────────────────────────────────────────────────
# PLOT 5 — Inference demo on test images
# Fix: rebuild model architecture then load weights only
# ─────────────────────────────────────────────────────────────

best_name  = df_backbone.loc[df_backbone['F1_Micro'].idxmax(), 'Run']

best_model = build_model(best_name, use_dropout=True, use_l2=True)
best_model.load_weights(f'partA_{best_name}_best.keras')

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for imgs, lbls in test_ds.take(1):
    preds = best_model.predict(imgs, verbose=0)
    for i, ax in enumerate(axes.flat):
        ax.imshow(imgs[i].numpy() / 255.0)
        gt   = [VOC_CLASSES[j] for j in range(NUM_CLASSES) if lbls[i][j] == 1]
        pred = [VOC_CLASSES[j] for j in range(NUM_CLASSES) if preds[i][j] >= THRESHOLD]
        match = set(gt) == set(pred)
        ax.set_title(f'GT: {", ".join(gt)}\nPred: {", ".join(pred)}',
                     fontsize=7.5, color='green' if match else 'red')
        ax.axis('off')

plt.suptitle(f'Inference — {best_name}  (green=exact match, red=mismatch)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('plot5_inference_demo.png', dpi=150, bbox_inches='tight')
plt.show()

del best_model; keras.backend.clear_session()

## Section 7 — Conclusion

In [ ]:
# ─────────────────────────────────────────────────────────────
# Conclusion
# Reads df_backbone and df_ablation, prints a full analysis
# ─────────────────────────────────────────────────────────────

import textwrap

SEP  = '=' * 68
sep  = '-' * 68
W    = 68

def box(title):
    print(SEP)
    print(f'  {title}')
    print(SEP)

def sub(title):
    print(f'\n{sep}')
    print(f'  {title}')
    print(sep)

# ══════════════════════════════════════════════════════════════
# PART A — BACKBONE COMPARISON ANALYSIS
# ══════════════════════════════════════════════════════════════
box('PART A — BACKBONE COMPARISON')

# Sort by F1_Micro descending
df_sorted = df_backbone.sort_values('F1_Micro', ascending=False).reset_index(drop=True)

best   = df_sorted.iloc[0]
second = df_sorted.iloc[1]
worst  = df_sorted.iloc[2]

print(f'\n  Ranking by F1-Micro (test set):')
for i, row in df_sorted.iterrows():
    bar = '█' * int(row['F1_Micro'] * 30)
    print(f'  #{i+1}  {row["Run"]:<14} F1={row["F1_Micro"]:.4f}  |{bar}')

print(f'\n  Best model  : {best["Run"]}')
print(f'  F1-Micro    : {best["F1_Micro"]:.4f}')
print(f'  F1-Macro    : {best["F1_Macro"]:.4f}')
print(f'  Precision   : {best["Precision"]:.4f}')
print(f'  Recall      : {best["Recall"]:.4f}')
print(f'  Accuracy    : {best["Accuracy"]:.4f}')
print(f'  Epochs run  : {int(best["Epochs_Run"])}  /  50  max')
print(f'  Train time  : {best["Time_min"]:.1f} min')

# Gap analysis
gap_1_2 = best['F1_Micro'] - second['F1_Micro']
gap_1_3 = best['F1_Micro'] - worst['F1_Micro']
print(f'\n  Gap: {best["Run"]} vs {second["Run"]:} = {gap_1_2:+.4f} F1-Micro')
print(f'  Gap: {best["Run"]} vs {worst["Run"]:} = {gap_1_3:+.4f} F1-Micro')

# Early Stopping observations
sub('Early Stopping Observations (Part A)')
for _, row in df_backbone.iterrows():
    ran_all = int(row['Epochs_Run']) == 50
    if ran_all:
        print(f'  {row["Run"]:<14}: ran ALL 50 epochs → val_loss kept improving → '
              f'no overfitting detected → Early Stopping correctly did NOT trigger')
    else:
        print(f'  {row["Run"]:<14}: stopped at epoch {int(row["Epochs_Run"])} '
              f'→ Early Stopping triggered → overfitting detected at that point')

# Architecture analysis
sub('Why did the models perform this way?')
backbone_notes = {
    'VGG16':
        ('138M parameters, simple stacked 3x3 convolutions. '
         'Feature extractor is very powerful but training the head is slower '
         'due to the large feature map size fed into GlobalAveragePooling.'),
    'ResNet50':
        ('25M parameters. Skip connections allow gradients to flow freely '
         'through 50 layers, avoiding vanishing gradient. With correct '
         'preprocess_input applied inside the model, frozen BatchNorm layers '
         'receive inputs in the exact distribution they were calibrated on.'),
    'MobileNetV2':
        ('3.4M parameters. Depthwise separable convolutions make it very '
         'efficient. preprocess_input scales pixels to [-1,1] — a clean '
         'normalized input that the frozen backbone handles well.'),
}
for name, note in backbone_notes.items():
    row = df_backbone[df_backbone['Run']==name].iloc[0]
    rank = df_sorted[df_sorted['Run']==name].index[0] + 1
    print(f'\n  {name} (Rank #{rank}, F1={row["F1_Micro"]:.4f}):')
    for line in textwrap.wrap(note, 62):
        print(f'    {line}')

# ══════════════════════════════════════════════════════════════
# PART B — ABLATION STUDY ANALYSIS
# ══════════════════════════════════════════════════════════════
box('\nPART B — ABLATION STUDY (ResNet50)')

# Baseline reference
abl_dict = {r['Run']: r for r in ablation_results}
baseline  = abl_dict['Exp1-Baseline']
full      = abl_dict['Exp5-FullModel']
drop_only = abl_dict['Exp2-DropoutOnly']
es_only   = abl_dict['Exp3-EarlyStpOnly']
l2_only   = abl_dict['Exp4-L2Only']

print(f'\n  Baseline F1-Micro  : {baseline["F1_Micro"]:.4f}  (no regularization)')
print(f'  Full model F1-Micro: {full["F1_Micro"]:.4f}  (Dropout + ES + L2)')
print(f'  Total improvement  : {full["F1_Micro"]-baseline["F1_Micro"]:+.4f}')

# Per-technique delta
sub('Effect of each technique (vs Baseline)')
experiments = [
    ('Dropout only',     drop_only),
    ('EarlyStp only',    es_only),
    ('L2 only',          l2_only),
    ('Full model',       full),
]
for name, row in experiments:
    delta = row['F1_Micro'] - baseline['F1_Micro']
    bar   = ('▲' if delta >= 0 else '▼') * max(1, int(abs(delta) * 40))
    sign  = '+' if delta >= 0 else ''
    print(f'  {name:<18}: F1={row["F1_Micro"]:.4f}  ({sign}{delta:.4f})  {bar}')

# Dropout effect
sub('Dropout — what the numbers say')
do_delta = drop_only['F1_Micro'] - baseline['F1_Micro']
if do_delta > 0.005:
    print(f'  Dropout improved F1 by {do_delta:+.4f}.')
    print('  Training curves should show a smaller train-val accuracy gap,')
    print('  confirming that random neuron disabling reduced co-adaptation.')
elif do_delta > -0.005:
    print(f'  Dropout had minimal effect ({do_delta:+.4f}).')
    print('  The model may not have been overfitting significantly with this head size.')
else:
    print(f'  Dropout hurt F1 by {do_delta:.4f}.')
    print('  This can occur when the model is underfitting — reducing capacity')
    print('  with dropout makes it harder to learn from limited signal.')

# Early Stopping effect
sub('Early Stopping — what the numbers say')
es_delta  = es_only['F1_Micro'] - baseline['F1_Micro']
es_epochs = int(es_only['Epochs_Run'])
if es_epochs < 50:
    print(f'  Early Stopping triggered at epoch {es_epochs}/50.')
    print(f'  F1 change vs baseline: {es_delta:+.4f}')
    print('  Training was halted before overfitting — restore_best_weights')
    print('  rewound model to the epoch with lowest val_loss.')
else:
    print(f'  Early Stopping did NOT trigger (ran all {es_epochs} epochs).')
    print(f'  F1 change vs baseline: {es_delta:+.4f}')
    print('  val_loss kept improving throughout — no overfitting occurred.')
    print('  This confirms the callback is working correctly; it simply was')
    print('  not needed because the model continued learning productively.')

# L2 effect
sub('Weight Decay (L2) — what the numbers say')
l2_delta = l2_only['F1_Micro'] - baseline['F1_Micro']
if l2_delta > 0.005:
    print(f'  L2 improved F1 by {l2_delta:+.4f}.')
    print('  Penalizing large weights smoothed the decision boundary and')
    print('  improved generalization to unseen test images.')
elif l2_delta > -0.005:
    print(f'  L2 had minimal isolated effect ({l2_delta:+.4f}).')
    print('  L2 benefits are often most visible in combination with other')
    print('  techniques rather than in isolation.')
else:
    print(f'  L2 alone hurt F1 by {l2_delta:.4f}.')
    print('  L2 increases training loss. Without Dropout or Early Stopping,')
    print('  the model cannot compensate and training is harder.')

# Full model verdict
sub('Full Model — all three combined')
full_delta = full['F1_Micro'] - baseline['F1_Micro']
if full_delta > 0:
    print(f'  Full model IMPROVED over baseline by {full_delta:+.4f} F1-Micro.')
    print('  This confirms Dropout, Early Stopping, and L2 are COMPLEMENTARY.')
    print('  Each technique addresses a different aspect of overfitting:')
    print('    Dropout   → prevents neuron co-adaptation')
    print('    EarlyStp  → prevents training past optimal epoch')
    print('    L2        → prevents weights from growing too large')
else:
    print(f'  Full model BELOW baseline by {full_delta:.4f} F1-Micro.')
    print('  This can occur when the model is underfitting — regularization')
    print('  reduces capacity further. Check if backbone features are properly')
    print('  normalized and if the training signal is strong enough.')

# ══════════════════════════════════════════════════════════════
# FINAL SUMMARY TABLE
# ══════════════════════════════════════════════════════════════
box('\nFINAL SUMMARY')
print(f'  Best backbone (Part A) : {best["Run"]}')
print(f'  Best F1-Micro          : {best["F1_Micro"]:.4f}')
print(f'  Best F1-Macro          : {best["F1_Macro"]:.4f}')
print(f'  Ablation winner        : {max(ablation_results, key=lambda x: x["F1_Micro"])["Run"]}')
print(f'  Ablation best F1       : {max(r["F1_Micro"] for r in ablation_results):.4f}')
print(f'  Ablation baseline F1   : {baseline["F1_Micro"]:.4f}')
total_gain = full['F1_Micro'] - baseline['F1_Micro']
print(f'  Regularization gain    : {total_gain:+.4f} (baseline → full model)')
print(f'\n  Framework : TensorFlow / Keras  |  GPU: Kaggle T4  |  SEED: 42')
print(f'  Dataset   : Pascal VOC 2012  |  Classes: 20  |  Task: Multi-Label')
print(SEP)


## Conclusion

---

### Part A — Backbone Comparison

Three pretrained CNN backbones were evaluated on Pascal VOC 2012 multi-label
image classification using transfer learning with full regularization
(Dropout + L2 + Early Stopping). Each backbone received images preprocessed
with its own official `preprocess_input` function applied inside the model,
ensuring frozen BatchNorm layers received inputs in the exact pixel distribution
used during ImageNet pre-training.

**Results (ranked by F1-Micro on test set):**

| Rank | Model | Accuracy | Precision | Recall | F1-Micro | F1-Macro |
|---|---|---|---|---|---|---|
| 🥇 1st | ResNet50 | 0.9710 | 0.8456 | 0.7609 | **0.8010** | 0.7840 |
| 🥈 2nd | MobileNetV2 | 0.9706 | 0.8580 | 0.7390 | 0.7940 | 0.7828 |
| 🥉 3rd | VGG16 | 0.9677 | 0.8217 | 0.7381 | 0.7776 | 0.7565 |

**Winner: ResNet50** with F1-Micro = 0.8010

**Why ResNet50 won:**
ResNet50's skip connections allow gradients to flow freely through all 50 layers,
avoiding the vanishing gradient problem that affects very deep plain networks like
VGG16. With correct `preprocess_input` applied inside the model, its frozen
BatchNorm layers received inputs in exactly the right distribution — producing
high-quality feature maps for the classification head to learn from.

**Why MobileNetV2 was competitive despite being the smallest model:**
MobileNetV2 has only 3.4M parameters but uses depthwise separable convolutions
that are highly efficient at extracting features. Its `preprocess_input` scales
pixels to [−1, 1] — a clean, symmetric range that works well with its inverted
residual blocks. Despite being 40× smaller than VGG16, it ranked second.

**Why VGG16 ranked last:**
VGG16 has 138M parameters — the largest of the three — but its simple stacked
3×3 convolution architecture without skip connections makes the feature extractor
less efficient at transfer learning compared to ResNet50. The large feature map
size fed into GlobalAveragePooling also reduces discriminative power.

**Early Stopping observation:**
All three models ran all 50 epochs without Early Stopping triggering. This is
not a bug — it means validation loss kept improving consistently throughout
training. There was no overfitting within the 50-epoch budget, confirming that
Dropout and L2 were successfully preventing overfitting and allowing the model
to keep learning productively up to epoch 50.

---

### Part B — Ablation Study (ResNet50)

To measure the individual contribution of each regularization technique,
ResNet50 was trained in 5 configurations — one technique added at a time.

**Results (vs Baseline F1-Micro = 0.7907):**

| Experiment | Dropout | Early Stopping | L2 | F1-Micro | Change |
|---|:---:|:---:|:---:|---|---|
| Exp1 — Baseline | ❌ | ❌ | ❌ | 0.7907 | reference |
| Exp2 — Dropout only | ✅ | ❌ | ❌ | **0.8070** | +0.0164 ✅ |
| Exp3 — Early Stopping only | ❌ | ✅ | ❌ | 0.7922 | +0.0015 ✅ |
| Exp4 — L2 only | ❌ | ❌ | ✅ | 0.7886 | −0.0020 |
| Exp5 — Full model | ✅ | ✅ | ✅ | 0.7987 | +0.0080 ✅ |

**Effect of Dropout (+0.0164 improvement):**
Dropout was the single most impactful regularization technique in isolation.
By randomly disabling 50% of Dense-512 neurons and 30% of Dense-256 neurons
each training step, it prevented neurons from co-adapting — forcing the network
to learn redundant, more general representations. The result was the highest
single-technique F1 of 0.8070, outperforming even the full model.

**Effect of Early Stopping (+0.0015 improvement, triggered at epoch 20):**
Early Stopping triggered at epoch 20 out of 50 (patience=7). The model had not
improved since epoch 13, so the callback automatically restored weights from
epoch 13 — the best validation loss checkpoint — preventing the model from
keeping 7 epochs of overfit weights. The improvement was modest (+0.0015)
because the model was not heavily overfitting, but the callback correctly
identified the optimal stopping point and rewound to it.

**Effect of Weight Decay / L2 (−0.0020 in isolation):**
L2 alone produced a very slight negative effect (−0.0020) compared to the
baseline. This is expected behaviour: L2 adds a penalty term to the loss
function that makes training harder by discouraging large weights. In isolation,
without Dropout preventing co-adaptation, the model's head struggles slightly
more to find a strong signal. However, L2 contributes positively when combined
with all three techniques in the full model.

**Full model — all three combined (+0.0080 improvement):**
The full model achieved F1-Micro = 0.7987, outperforming the baseline by +0.0080.
This confirms that Dropout, Early Stopping, and L2 are complementary techniques
that each address a different aspect of overfitting:
- **Dropout** prevents neuron co-adaptation during training
- **Early Stopping** prevents training past the optimal epoch
- **L2** prevents individual weights from growing too large

**Notable finding — Dropout only (0.8070) outperformed the Full model (0.7987):**
This is an explainable and important result. Dropout is the most impactful single
technique on this dataset. Adding L2 regularization on top introduces additional
training difficulty through weight penalties. On a dataset of ~8,500 training
images, the model has limited capacity to absorb both the Dropout noise and the
L2 penalty simultaneously within 50 epochs. This does not mean the Full model
is wrong — it still outperforms the baseline by +0.0080 — but it shows that
regularization technique combinations must be tuned to the dataset size and
training budget.

---

### Key Technical Finding

The most important implementation insight from this project is that
**backbone-specific preprocessing is critical for correct transfer learning**.
In an earlier experiment using uniform `/255` normalization for all backbones,
ResNet50 achieved only F1-Micro = 0.149 — barely above random chance — because
its frozen BatchNorm layers received inputs outside the calibrated range they
were trained on during ImageNet pre-training. After applying
`resnet50.preprocess_input()` (channel-wise mean subtraction) inside the model
as a dedicated layer, ResNet50 became the best-performing backbone at
F1-Micro = 0.8010. This demonstrates that understanding how pretrained models
expect their inputs is as important as the model architecture itself.

---

### Final Summary

| Metric | Value |
|---|---|
| Best backbone | ResNet50 |
| Best F1-Micro (Part A) | 0.8010 |
| Best F1-Macro (Part A) | 0.7840 |
| Best ablation config | Dropout only — F1 = 0.8070 |
| Full model gain over baseline | +0.0080 |
| Early Stopping best epoch | Epoch 13 (triggered at epoch 20) |
| Dataset | Pascal VOC 2012 — 20 classes — Multi-label |
| Framework | TensorFlow / Keras — GPU: Kaggle T4 — Seed: 42 |

---
*Computer Vision Project — BS Computer Science, Semester VIII*